In [94]:
#from pyforest import *
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, StackingClassifier
from sklearn.svm import SVR
from sklearn.metrics import  mean_squared_error, r2_score, confusion_matrix, classification_report
#from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('final_solar_dataset.csv')

In [3]:
df = df.dropna()
#Noticed a few nan values which might be due to merging of the files, so i dropped them since they are few
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3157 entries, 0 to 3181
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DC_POWER             3157 non-null   float64
 1   AC_POWER             3157 non-null   float64
 2   HOUR                 3157 non-null   float64
 3   AMBIENT_TEMPERATURE  3157 non-null   float64
 4   MODULE_TEMPERATURE   3157 non-null   float64
 5   IRRADIATION          3157 non-null   float64
dtypes: float64(6)
memory usage: 172.6 KB


THE AIM OF THE MODEL IS MAKE PREDICTIVE MAINTENANCE SYSTEM AND TO SO, WE NEED TO KNOW THE SYSTEM EFFICIENCY AND PERFOMANCE.

I WILL BE USING THE PERFORMANCE RATIO, THIS SHOWS THE SYSTEM OVERALL WORKING RATE, THIS IS CALCULATED BY DIVIDING THE AC_POWER BY IRRADIATION.
THE IRRADIATION IS THE AMOUNT OF SUNLIGHT THE PANEL IS TRAPPING, AND THE AC_POWER IS THE INVERTER POWER I.E THE AMOUNT STORED/AMOUNT USED.

In [4]:
df['PERFORMANCE_RATIO'] = df['AC_POWER'] / (df['IRRADIATION'] + 0.000001)
#Added 0.000001 to the Irradiation column due to the fact that some values are zero and 
# dividing by zero won't give areasonable value

In [5]:
df.describe()

,DC_POWER,AC_POWER,HOUR,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,PERFORMANCE_RATIO
count,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000
mean,3117.309001,304.857459,11.626227,25.560257,31.175453,0.230103,711.839438
std,4002.425859,391.090163,6.867083,3.351059,12.272685,0.301348,673.004890
min,0.000000,0.000000,0.000000,20.398505,18.140415,0.000000,0.000000
25%,0.000000,0.000000,6.000000,22.739895,21.130249,0.000000,0.000000
50%,383.190747,37.040016,12.000000,24.680324,24.801971,0.027748,1167.203534
75%,6382.267857,625.096023,18.000000,27.941221,41.449481,0.451576,1359.653715
max,13588.081169,1325.009659,23.000000,35.252486,65.545714,1.221652,2000.844738


In [6]:
#seperating the datasets into train and test to prepare it for the model

y = df['PERFORMANCE_RATIO']
x = df.drop('PERFORMANCE_RATIO', axis = 1)


In [7]:
#Tried various techniques which include linear regression, randomforest regressor and SVM
df['PERFORMANCE_RATIO'].info()

<class 'pandas.core.series.Series'>
Index: 3157 entries, 0 to 3181
Series name: PERFORMANCE_RATIO
Non-Null Count  Dtype  
--------------  -----  
3157 non-null   float64
dtypes: float64(1)
memory usage: 49.3 KB


In [8]:
#FOR LINEAR REGRESSION
linear_model = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('regressor' ,  LinearRegression())

]) 
linear_model.fit(x, y)
linear_model_pred = linear_model.predict(x)





#FOR RANDOMFOREST REGRESSOR
randomforest_model = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor())
])
randomforest_model.fit(x, y)
randomforest_model_pred = randomforest_model.predict(x)





#FOR SVM
svr_model = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('SVR', SVR(kernel='rbf'))
])
svr_model.fit(x, y)
svr_model_pred = svr_model.predict(x)


In [9]:
results= pd.DataFrame({
    'Actual' : y,
    'linear_pred' : linear_model_pred,
    'Random_pred' : randomforest_model_pred,
    'Svr_Pred' : svr_model_pred
})

In [10]:
#Added 0.0000001 to the predicted column which is almost insignificant and this is due to the fact that some values of the 
#actual column has the value of zero which in return will trigger cases of nan values.

results['linear_performance_drop'] =abs(results['Actual'] - results['linear_pred']) / (results['linear_pred'] + 0.0000001)
results['random_performance_drop'] =(results['Actual'] - results['Random_pred']) / (results['Random_pred'] + 0.0000001)
results['svr_performance_drop'] =abs(results['Actual'] - results['Svr_Pred']) / (results['Svr_Pred'] + 0.0000001) 
results

,Actual,linear_pred,Random_pred,Svr_Pred,linear_performance_drop,random_performance_drop,svr_performance_drop
0,0.0,391.482873,0.0,247.179423,1.0,0.0,1.0
1,0.0,387.127772,0.0,241.548334,1.0,0.0,1.0
2,0.0,380.340094,0.0,233.178245,1.0,0.0,1.0
3,0.0,374.836614,0.0,227.385345,1.0,0.0,1.0
4,0.0,361.012316,0.0,206.846589,1.0,0.0,1.0
...,...,...,...,...,...,...,...
3177,0.0,182.565326,0.0,163.753003,1.0,0.0,1.0
3178,0.0,176.638727,0.0,176.070664,1.0,0.0,1.0
3179,0.0,165.172401,0.0,174.622050,1.0,0.0,1.0
3180,0.0,164.168236,0.0,175.072043,1.0,0.0,1.0


In [11]:
#print('svr_rmse = ', root_mean_squared_error(y, svr_model_pred))
print('svr_mse = ', mean_squared_error(y, svr_model_pred))
print('svr_r2 =' , r2_score(y, svr_model_pred))

svr_mse =  173161.62119069867
svr_r2 = 0.6175692621672524


In [12]:
#print('RF_rmse = ', root_mean_squared_error(y, randomforest_model_pred))
print('RF_mse = ', mean_squared_error(y, randomforest_model_pred))
print('RF_r2 = ', r2_score(y,randomforest_model_pred))

RF_mse =  222.68319289663467
RF_r2 =  0.9995081999280393


In [13]:
#print('Linear_rmse = ', root_mean_squared_error(y, linear_model_pred))
print('Linear_mse = ', mean_squared_error(y, linear_model_pred))
print('Linear_r2 = ', r2_score(y, linear_model_pred))

Linear_mse =  165558.27855058046
Linear_r2 =  0.6343613891747337


from the models, Randomforest performed well,so will be using it

In [14]:
results.describe()

,Actual,linear_pred,Random_pred,Svr_Pred,linear_performance_drop,random_performance_drop,svr_performance_drop
count,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000
mean,711.839438,711.839438,712.077160,584.277867,0.878084,-0.002056,1.050256
std,673.004890,536.026907,672.352498,478.289084,0.997770,0.043332,1.484497
min,0.000000,114.775278,0.000000,98.888551,0.000383,-1.000000,0.000067
25%,0.000000,251.187480,0.000000,155.033162,0.129830,-0.000329,0.180336
50%,1167.203534,373.677667,1183.953294,318.446978,1.000000,0.000000,1.000000
75%,1359.653715,1296.603007,1360.436718,1135.230198,1.000000,0.000796,1.000000
max,2000.844738,2926.983858,1775.572045,1342.146359,7.791161,0.375894,11.218868


In [15]:
df['PREDICTED_PERFORMANCE'] = results['Random_pred']
df['PERFORMANCE_DROP'] = results['random_performance_drop']
#using the predicted performance from the randomforest model
df

,DC_POWER,AC_POWER,HOUR,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,PERFORMANCE_RATIO,PREDICTED_PERFORMANCE,PERFORMANCE_DROP
0,0.0,0.0,0.0,25.184316,22.857507,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,25.084589,22.761668,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,24.935753,22.592306,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,24.846130,22.360852,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,24.621525,22.165423,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
3177,0.0,0.0,22.0,22.150570,21.480377,0.0,0.0,0.0,0.0
3178,0.0,0.0,23.0,22.129816,21.389024,0.0,0.0,0.0,0.0
3179,0.0,0.0,23.0,22.008275,20.709211,0.0,0.0,0.0,0.0
3180,0.0,0.0,23.0,21.969495,20.734963,0.0,0.0,0.0,0.0


In [16]:
df.describe()

,DC_POWER,AC_POWER,HOUR,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,PERFORMANCE_RATIO,PREDICTED_PERFORMANCE,PERFORMANCE_DROP
count,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000,3157.000000
mean,3117.309001,304.857459,11.626227,25.560257,31.175453,0.230103,711.839438,712.077160,-0.002056
std,4002.425859,391.090163,6.867083,3.351059,12.272685,0.301348,673.004890,672.352498,0.043332
min,0.000000,0.000000,0.000000,20.398505,18.140415,0.000000,0.000000,0.000000,-1.000000
25%,0.000000,0.000000,6.000000,22.739895,21.130249,0.000000,0.000000,0.000000,-0.000329
50%,383.190747,37.040016,12.000000,24.680324,24.801971,0.027748,1167.203534,1183.953294,0.000000
75%,6382.267857,625.096023,18.000000,27.941221,41.449481,0.451576,1359.653715,1360.436718,0.000796
max,13588.081169,1325.009659,23.000000,35.252486,65.545714,1.221652,2000.844738,1775.572045,0.375894


In [18]:
#Building the classifier target, the aim is to create a 
# target column to fit in the fault classification of the system
#based on the performance drop

def fault_classifier(x):
    
    if x >= 0:
        return 'Normal'
    else:
        return 'Abnormal'
    

df['FAULT_STATUS'] = df['PERFORMANCE_DROP'].apply(fault_classifier)
df.head(50)

,DC_POWER,AC_POWER,HOUR,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,PERFORMANCE_RATIO,PREDICTED_PERFORMANCE,PERFORMANCE_DROP,FAULT_STATUS
0,0.000000,0.000000,0.0,25.184316,22.857507,0.000000,0.000000,0.000000,0.000000,Normal
1,0.000000,0.000000,0.0,25.084589,22.761668,0.000000,0.000000,0.000000,0.000000,Normal
2,0.000000,0.000000,0.0,24.935753,22.592306,0.000000,0.000000,0.000000,0.000000,Normal
3,0.000000,0.000000,0.0,24.846130,22.360852,0.000000,0.000000,0.000000,0.000000,Normal
4,0.000000,0.000000,1.0,24.621525,22.165423,0.000000,0.000000,0.000000,0.000000,Normal
5,0.000000,0.000000,1.0,24.536092,21.968571,0.000000,0.000000,0.000000,0.000000,Normal
6,0.000000,0.000000,1.0,24.638674,22.352926,0.000000,0.000000,0.000000,0.000000,Normal
7,0.000000,0.000000,1.0,24.873022,23.160919,0.000000,0.000000,0.000000,0.000000,Normal
8,0.000000,0.000000,2.0,24.936930,23.026113,0.000000,0.000000,0.000000,0.000000,Normal
9,0.000000,0.000000,2.0,25.012248,23.343229,0.000000,0.000000,0.000000,0.000000,Normal


In [19]:
df['FAULT_STATUS'].value_counts()
#The data obviously looks inbalnaced, it contains more normal features than the Abnormal Feature 


FAULT_STATUS
Normal      2339
Abnormal     818
Name: count, dtype: int64

In [20]:
classifier_df = df.drop(['PREDICTED_PERFORMANCE', 'PERFORMANCE_DROP', 'PERFORMANCE_RATIO'], axis =1 )

In [21]:
X = classifier_df.drop(['FAULT_STATUS'], axis =1) 
Y = classifier_df['FAULT_STATUS']


In [22]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3)

In [117]:
random_model = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('regressor', RandomForestClassifier(class_weight='balanced'))
])
random_model.fit(x_train, y_train)
random_model_pred = random_model.predict(x_test)

In [118]:
random_model_pred

array(['Normal', 'Abnormal', 'Normal', 'Abnormal', 'Abnormal', 'Normal',
       'Normal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal',
       'Abnormal', 'Normal', 'Normal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Normal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Abnormal',
       'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Abnormal',
       'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Normal', 'Normal', 'Abnormal', 'Abnormal', 'Abnormal',
       'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal',
       'Normal', '

In [119]:
random_model_data = pd.DataFrame(random_model_pred, columns=['RandomForest'])
random_model_data.value_counts()

RandomForest
Normal          710
Abnormal        238
Name: count, dtype: int64

In [120]:
print(classification_report(y_test, random_model_pred))

#from this the results, the model predicts more of the normal class, which makes it biased

              precision    recall  f1-score   support

    Abnormal       0.56      0.55      0.55       244
      Normal       0.84      0.85      0.85       704

    accuracy                           0.77       948
   macro avg       0.70      0.70      0.70       948
weighted avg       0.77      0.77      0.77       948



In [121]:
from imblearn.ensemble import EasyEnsembleClassifier

easy_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    
    ('model', EasyEnsembleClassifier())
])

easy_pipeline.fit(x_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', EasyEnsembleClassifier())])

In [122]:
easy_pipeline_pred = easy_pipeline.predict(x_test)

In [123]:
easy_pipeline_pred

array(['Abnormal', 'Abnormal', 'Normal', 'Abnormal', 'Abnormal', 'Normal',
       'Normal', 'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal',
       'Abnormal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal',
       'Abnormal', 'Normal', 'Abnormal', 'Abnormal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Abnormal',
       'Abnormal', 'Normal', 'Normal', 'Abnormal', 'Abnormal', 'Abnormal',
       'Normal', 'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal',
       'Normal', 'Abnormal', 'Abnormal', 'Normal', 'Abnormal', 'Normal',
       'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal', 'Abnormal',
       'Normal', 'Abnormal', 'Normal', 'Abnormal', 'Normal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal', 'N

In [124]:
easy_data = pd.DataFrame(easy_pipeline_pred, columns=['EasyEnsembleClassifier'])
easy_data.value_counts()

EasyEnsembleClassifier
Abnormal                  494
Normal                    454
Name: count, dtype: int64

In [125]:
print(classification_report(y_test, easy_pipeline_pred))

              precision    recall  f1-score   support

    Abnormal       0.48      0.98      0.64       244
      Normal       0.99      0.64      0.77       704

    accuracy                           0.72       948
   macro avg       0.73      0.81      0.71       948
weighted avg       0.86      0.72      0.74       948



In [126]:
print(confusion_matrix(y_test, easy_pipeline_pred))

[[238   6]
 [256 448]]


In [127]:
from imblearn.ensemble import BalancedRandomForestClassifier

balanced_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    
    ('model', BalancedRandomForestClassifier())
])

balanced_pipeline.fit(x_train, y_train)

c:\Users\USER\Desktop\Solar_MAintenance_App\Solar\Lib\site-packages\imblearn\ensemble\_forest.py:546: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\USER\Desktop\Solar_MAintenance_App\Solar\Lib\site-packages\imblearn\ensemble\_forest.py:558: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(


Pipeline(steps=[('scaler', StandardScaler()),
                ('model', BalancedRandomForestClassifier())])

In [128]:
balanced_pipeline_pred = balanced_pipeline.predict(x_test)

In [129]:
balanced_pipeline_pred

array(['Abnormal', 'Abnormal', 'Normal', 'Abnormal', 'Abnormal', 'Normal',
       'Normal', 'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal',
       'Abnormal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal',
       'Normal', 'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal',
       'Abnormal', 'Normal', 'Abnormal', 'Abnormal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Abnormal',
       'Abnormal', 'Normal', 'Normal', 'Abnormal', 'Abnormal', 'Abnormal',
       'Normal', 'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Abnormal',
       'Normal', 'Abnormal', 'Abnormal', 'Normal', 'Abnormal', 'Normal',
       'Normal', 'Abnormal', 'Normal', 'Normal', 'Normal', 'Abnormal',
       'Normal', 'Abnormal', 'Normal', 'Abnormal', 'Normal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal', 'Abnormal',
       'Abnormal', 'Abnormal', 'Abnormal', 'Normal', 'Normal', 

In [130]:
balanced_data = pd.DataFrame(balanced_pipeline_pred, columns=['Classifier'])
balanced_data.value_counts()

Classifier
Abnormal      474
Normal        474
Name: count, dtype: int64

In [132]:
print(classification_report(y_test, balanced_pipeline_pred))

              precision    recall  f1-score   support

    Abnormal       0.49      0.96      0.65       244
      Normal       0.98      0.66      0.79       704

    accuracy                           0.74       948
   macro avg       0.74      0.81      0.72       948
weighted avg       0.85      0.74      0.75       948



In [133]:
print(confusion_matrix(y_test, balanced_pipeline_pred))

[[234  10]
 [240 464]]


In [134]:
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model' , BalancedBaggingClassifier(
        base_estimator=DecisionTreeClassifier(class_weight= 'balanced'),
        n_estimators= 10,
        random_state=42
    ))
])

bag_pipeline.fit(x_train, y_train)

c:\Users\USER\Desktop\Solar_MAintenance_App\Solar\Lib\site-packages\imblearn\ensemble\_bagging.py:362: FutureWarning: `base_estimator` was renamed to `estimator` in version 0.10 and will be removed in 0.12.
  warnings.warn(


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 BalancedBaggingClassifier(base_estimator=DecisionTreeClassifier(class_weight='balanced'),
                                           random_state=42))])

In [135]:
bag_pipeline_pred = bag_pipeline.predict(x_test)


bag_data = pd.DataFrame(bag_pipeline_pred, columns=['Classifier'])
bag_data.value_counts()

Classifier
Normal        536
Abnormal      412
Name: count, dtype: int64

In [137]:
print(classification_report(y_test, bag_pipeline_pred))

              precision    recall  f1-score   support

    Abnormal       0.51      0.86      0.64       244
      Normal       0.93      0.71      0.81       704

    accuracy                           0.75       948
   macro avg       0.72      0.78      0.72       948
weighted avg       0.82      0.75      0.76       948



In [138]:
print(confusion_matrix(y_test, bag_pipeline_pred))

[[209  35]
 [203 501]]


In [139]:
y_ = pd.DataFrame(classifier_df['FAULT_STATUS'])
y_

,FAULT_STATUS
0,Normal
1,Normal
2,Normal
3,Normal
4,Normal
...,...
3177,Normal
3178,Normal
3179,Normal
3180,Normal


In [140]:
"""encode = OneHotEncoder(sparse=False, handle_unknown='ignore')

y_encode = encode.fit_transform(y_)

X_train, X_test, Y_train, Y_test = train_test_split(X, y_encode, test_size=.3, random_state=42)
y_encode
"""

"encode = OneHotEncoder(sparse=False, handle_unknown='ignore')\n\ny_encode = encode.fit_transform(y_)\n\nX_train, X_test, Y_train, Y_test = train_test_split(X, y_encode, test_size=.3, random_state=42)\ny_encode\n"

In [141]:
le = LabelEncoder()

y_encode = le.fit_transform(y_)
X_train, X_test, Y_train, Y_test = train_test_split(X, y_encode, test_size=.30, random_state=42)

c:\Users\USER\Desktop\Solar_MAintenance_App\Solar\Lib\site-packages\sklearn\preprocessing\_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [142]:
from xgboost import XGBClassifier

XGB_model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        objective ='binary:logistic',
        eval_metric='logloss',
        learning_rate=0.1,
        max_depth=5,
        n_estimators = 200,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )) 
])



XGB_model.fit(X_train, Y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [143]:
XGB_model_pred = XGB_model.predict(X_test)

#XGB_model_pred = np.argmax(XGB_model_pred, axis=1)

print(classification_report(Y_test, XGB_model_pred))
#print(Y_test.shape)
#print(XGB_model_pred.shape)

              precision    recall  f1-score   support

           0       0.57      0.61      0.59       230
           1       0.87      0.85      0.86       718

    accuracy                           0.79       948
   macro avg       0.72      0.73      0.73       948
weighted avg       0.80      0.79      0.80       948



In [87]:
print(confusion_matrix(Y_test, XGB_model_pred))

[[141  89]
 [106 612]]


In [144]:
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(
    class_weight='balanced',
    y= Y_train
)

XGB_model.fit(X_train, Y_train, model__sample_weight=weights)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [145]:
XGB_model_pred_2 = XGB_model.predict(X_test)

#XGB_model_pred_2 = np.argmax(XGB_model_pred_2, axis=1)

print(classification_report(Y_test, XGB_model_pred_2))

              precision    recall  f1-score   support

           0       0.50      0.83      0.63       230
           1       0.93      0.74      0.82       718

    accuracy                           0.76       948
   macro avg       0.72      0.78      0.72       948
weighted avg       0.83      0.76      0.78       948



In [146]:
originals = le.inverse_transform(XGB_model_pred)

In [147]:
original = pd.DataFrame(originals, columns=['Original'])
encoded = pd.DataFrame(XGB_model_pred, columns= ['Encoded'])


encoded['Original'] = original['Original']
encoded.head(50)

,Encoded,Original
0,1,Normal
1,1,Normal
2,1,Normal
3,0,Abnormal
4,1,Normal
5,1,Normal
6,1,Normal
7,1,Normal
8,1,Normal
9,1,Normal


In [148]:
logreg = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced'
)


xgb_log = StackingClassifier(
    estimators=[('xgb', XGB_model)],
    final_estimator=logreg,
    passthrough=True
)

xgb_log.fit(X_train, Y_train)

StackingClassifier(estimators=[('xgb',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('model',
                                                 XGBClassifier(base_score=None,
                                                               booster=None,
                                                               callbacks=None,
                                                               colsample_bylevel=None,
                                                               colsample_bynode=None,
                                                               colsample_bytree=0.8,
                                                               device=None,
                                                               early_stopping_rounds=None,
                                                               enable_categorical=False,
                                                               eval_metric='logloss',
                                                               feature_types=None,
                                                               feature_weights=None,
                                                               gamma=Non...
                                                               learning_rate=0.1,
                                                               max_bin=None,
                                                               max_cat_threshold=None,
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=5,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=200,
                                                               n_jobs=None,
                                                               num_parallel_tree=None, ...))]))],
                   final_estimator=LogisticRegression(class_weight='balanced',
                                                      max_iter=1000),
                   passthrough=True)

In [149]:
xgb_log_pred = xgb_log.predict(X_test)

In [150]:
print(classification_report(Y_test,xgb_log_pred))

              precision    recall  f1-score   support

           0       0.48      0.82      0.61       230
           1       0.92      0.72      0.81       718

    accuracy                           0.74       948
   macro avg       0.70      0.77      0.71       948
weighted avg       0.82      0.74      0.76       948

